In [61]:
# 批量文件转yolo格式
import os
import json
import shutil
import numpy as np
from tqdm import tqdm

In [108]:
os.chdir('../../') 
# os.chdir('pyproject/ultralytics-main')
current_dir = os.getcwd()  # 获取当前工作目录
print("当前工作目录:", current_dir)

当前工作目录: e:\pyproject\ultralytics-main


In [109]:
Dataset_root = 'datasets/pose12'

In [110]:
# 框的类别
bbox_class = {
    'person':0  
}

# 关键点的类别
keypoint_class = ['head', 'l_shoulder', 'r_shoulder', 'l_elbow', 'r_elbow', 'l_wrist', 'r_wrist', 'l_hip', 'r_hip', 'l_knee', 'r_knee', 'l_ankle', 'r_ankle']

In [111]:
os.chdir(Dataset_root)

In [112]:
os.mkdir('labels_txt')
os.mkdir('labels_txt/train')
os.mkdir('labels_txt/val')

In [113]:
# os.chdir('..')
# os.chdir('..')
current_dir = os.getcwd()  # 获取当前工作目录
print("当前工作目录:", current_dir)

当前工作目录: e:\pyproject\ultralytics-main\datasets\pose12


In [114]:
# 函数-处理单个labelme标注json文件
def process_single_json(labelme_path, save_folder='../../labels_txt/train'):
    
    with open(labelme_path, 'r', encoding='utf-8') as f:
        labelme = json.load(f)

    img_width = labelme['imageWidth']   # 图像宽度
    img_height = labelme['imageHeight'] # 图像高度

    # 生成 YOLO 格式的 txt 文件
    suffix = labelme_path.split('.')[-2]
    yolo_txt_path = suffix + '.txt'

    with open(yolo_txt_path, 'w', encoding='utf-8') as f:

        for each_ann in labelme['shapes']: # 遍历每个标注

            if each_ann['shape_type'] == 'rectangle': # 每个框，在 txt 里写一行

                yolo_str = ''

                ## 框的信息
                # 框的类别 ID
                bbox_class_id = bbox_class[each_ann['label']]
                yolo_str += '{} '.format(bbox_class_id)
                # 左上角和右下角的 XY 像素坐标
                bbox_top_left_x = int(min(each_ann['points'][0][0], each_ann['points'][1][0]))
                bbox_bottom_right_x = int(max(each_ann['points'][0][0], each_ann['points'][1][0]))
                bbox_top_left_y = int(min(each_ann['points'][0][1], each_ann['points'][1][1]))
                bbox_bottom_right_y = int(max(each_ann['points'][0][1], each_ann['points'][1][1]))
                # 框中心点的 XY 像素坐标
                bbox_center_x = int((bbox_top_left_x + bbox_bottom_right_x) / 2)
                bbox_center_y = int((bbox_top_left_y + bbox_bottom_right_y) / 2)
                # 框宽度
                bbox_width = bbox_bottom_right_x - bbox_top_left_x
                # 框高度
                bbox_height = bbox_bottom_right_y - bbox_top_left_y
                # 框中心点归一化坐标
                bbox_center_x_norm = bbox_center_x / img_width
                bbox_center_y_norm = bbox_center_y / img_height
                # 框归一化宽度
                bbox_width_norm = bbox_width / img_width
                # 框归一化高度
                bbox_height_norm = bbox_height / img_height

                yolo_str += '{:.5f} {:.5f} {:.5f} {:.5f} '.format(bbox_center_x_norm, bbox_center_y_norm, bbox_width_norm, bbox_height_norm)

                ## 找到该框中所有关键点，存在字典 bbox_keypoints_dict 中
                bbox_keypoints_dict = {}
                for each_ann in labelme['shapes']: # 遍历所有标注
                    if each_ann['shape_type'] == 'point': # 筛选出关键点标注
                        # 关键点XY坐标、类别
                        x = int(each_ann['points'][0][0])
                        y = int(each_ann['points'][0][1])
                        label = each_ann['label']
                        if (x>bbox_top_left_x) & (x<bbox_bottom_right_x) & (y<bbox_bottom_right_y) & (y>bbox_top_left_y): # 筛选出在该个体框中的关键点
                            bbox_keypoints_dict[label] = [x, y]

                ## 把关键点按顺序排好
                for each_class in keypoint_class: # 遍历每一类关键点
                    if each_class in bbox_keypoints_dict:
                        keypoint_x_norm = bbox_keypoints_dict[each_class][0] / img_width
                        keypoint_y_norm = bbox_keypoints_dict[each_class][1] / img_height
                        yolo_str += '{:.5f} {:.5f} {} '.format(keypoint_x_norm, keypoint_y_norm, 2) # 2-可见不遮挡 1-遮挡 0-没有点
                    else: # 不存在的点，一律为0
                        yolo_str += '0 0 0 '
                # 写入 txt 文件中
                f.write(yolo_str + '\n')
                
    shutil.move(yolo_txt_path, save_folder)
    print('{} --> {} 转换完成'.format(labelme_path, yolo_txt_path))   

In [115]:
# 转换训练集标注文件至labels/train目录
os.chdir('labels/train')

save_folder = '../../labels_txt/train'
for labelme_path in os.listdir():
    try:
        process_single_json(labelme_path, save_folder=save_folder)
    except:
        print('******有误******', labelme_path)
print('YOLO格式的txt标注文件已保存至 ', save_folder)

0107-000290-01.json --> 0107-000290-01.txt 转换完成
0107-000291-01.json --> 0107-000291-01.txt 转换完成
0107-000293-01.json --> 0107-000293-01.txt 转换完成
0107-000304-11.json --> 0107-000304-11.txt 转换完成
0107-000325-11.json --> 0107-000325-11.txt 转换完成
0107-000344-11.json --> 0107-000344-11.txt 转换完成
0107-000364-11.json --> 0107-000364-11.txt 转换完成
0107-000382-11.json --> 0107-000382-11.txt 转换完成
0107-000383-11.json --> 0107-000383-11.txt 转换完成
0107-000403-11.json --> 0107-000403-11.txt 转换完成
0107-000404-11.json --> 0107-000404-11.txt 转换完成
0107-000423-11.json --> 0107-000423-11.txt 转换完成
0107-000445-11.json --> 0107-000445-11.txt 转换完成
0107-000461-11.json --> 0107-000461-11.txt 转换完成
0107-000462-11.json --> 0107-000462-11.txt 转换完成
0107-000480-11.json --> 0107-000480-11.txt 转换完成
0107-000503-11.json --> 0107-000503-11.txt 转换完成
0107-000504-11.json --> 0107-000504-11.txt 转换完成
0107-000525-11.json --> 0107-000525-11.txt 转换完成
0107-000545-11.json --> 0107-000545-11.txt 转换完成
0107-000582-11.json --> 0107-000582-11.t

In [116]:
os.chdir('../../')
current_dir = os.getcwd()  # 获取当前工作目录
print("当前工作目录:", current_dir)

当前工作目录: e:\pyproject\ultralytics-main\datasets\pose12


In [117]:
# 转换测试集标注文件至labels/val目录
os.chdir('labels/val')

save_folder = '../../labels_txt/val'
for labelme_path in os.listdir():
    try:
        process_single_json(labelme_path, save_folder=save_folder)
    except:
        print('******有误******', labelme_path)
print('YOLO格式的txt标注文件已保存至 ', save_folder)

0107-000292-01.json --> 0107-000292-01.txt 转换完成
0107-000294-01.json --> 0107-000294-01.txt 转换完成
0107-000343-11.json --> 0107-000343-11.txt 转换完成
0107-000501-11.json --> 0107-000501-11.txt 转换完成
0107-000502-11.json --> 0107-000502-11.txt 转换完成
0107-000580-11.json --> 0107-000580-11.txt 转换完成
0107-000581-11.json --> 0107-000581-11.txt 转换完成
0107-000613-11.json --> 0107-000613-11.txt 转换完成
0107-000619-01.json --> 0107-000619-01.txt 转换完成
0107-000620-01.json --> 0107-000620-01.txt 转换完成
0107-000641-11.json --> 0107-000641-11.txt 转换完成
0107-000647-11.json --> 0107-000647-11.txt 转换完成
0107-000655-11.json --> 0107-000655-11.txt 转换完成
0107-000675-01.json --> 0107-000675-01.txt 转换完成
0107-000676-01.json --> 0107-000676-01.txt 转换完成
0107-000677-01.json --> 0107-000677-01.txt 转换完成
0107-000705-01.json --> 0107-000705-01.txt 转换完成
0107-000707-01.json --> 0107-000707-01.txt 转换完成
0107-000708-01.json --> 0107-000708-01.txt 转换完成
0107-000724-11.json --> 0107-000724-11.txt 转换完成
0107-000735-01.json --> 0107-000735-01.t

In [15]:
os.chdir('../')

In [14]:
# 得到YOLO格式的关键点检测数据集
import seedir as sd
sd.seedir('SJB_25_Dataset', style='emoji', depthlimit=2)

📁 SJB_25_Dataset/
├─📁 images/
│ ├─📁 train/
│ └─📁 val/
└─📁 labels/
  ├─📁 train/
  └─📁 val/
